In [2]:
import bilby as bb
import gwpopulation as gwpop
import jax
import matplotlib.pyplot as plt
import pandas as pd
from bilby.core.prior import PriorDict, Uniform
from bilby.hyper.model import Model
from gwpopulation.experimental.jax import JittedLikelihood

gwpop.set_backend("jax")

xp = gwpop.utils.xp

%matplotlib inline

In [3]:
posteriors = pd.read_pickle("gwtc-3-samples.pkl")
del posteriors[15]
del posteriors[38]

In [4]:
import dill

with open("gwtc-3-injections.pkl", "rb") as ff:
    injections = dill.load(ff)

In [5]:
model = Model(
    model_functions=[
        gwpop.models.mass.two_component_primary_mass_ratio,
        gwpop.models.spin.iid_spin,
        gwpop.models.redshift.PowerLawRedshift(cosmo_model="Planck15"),
    ],
    cache=False,
)

vt = gwpop.vt.ResamplingVT(model=model, data=injections, n_events=len(posteriors))

likelihood = gwpop.hyperpe.HyperparameterLikelihood(
    posteriors=posteriors,
    hyper_prior=model,
    selection_function=vt,
)

In [ ]:
priors = PriorDict()

# mass
priors["alpha"] = Uniform(minimum=-2, maximum=4, latex_label="$\\alpha$")
priors["beta"] = Uniform(minimum=-4, maximum=12, latex_label="$\\beta$")
priors["mmin"] = Uniform(minimum=2, maximum=2.5, latex_label="$m_{\\min}$")
priors["mmax"] = Uniform(minimum=80, maximum=100, latex_label="$m_{\\max}$")
priors["lam"] = Uniform(minimum=0, maximum=1, latex_label="$\\lambda_{m}$")
priors["mpp"] = Uniform(minimum=10, maximum=50, latex_label="$\\mu_{m}$")
priors["sigpp"] = Uniform(minimum=1, maximum=10, latex_label="$\\sigma_{m}$")
priors["gaussian_mass_maximum"] = 100
# spin
priors["amax"] = 1
priors["alpha_chi"] = Uniform(minimum=1, maximum=6, latex_label="$\\alpha_{\\chi}$")
priors["beta_chi"] = Uniform(minimum=1, maximum=6, latex_label="$\\beta_{\\chi}$")
priors["xi_spin"] = Uniform(minimum=0, maximum=1, latex_label="$\\xi$")
priors["sigma_spin"] = Uniform(minimum=0.3, maximum=4, latex_label="$\\sigma$")

priors["lamb"] = Uniform(minimum=-1, maximum=10, latex_label="$\\lambda_{z}$")

In [9]:
import numpy as np
from gwpopulation.models.redshift import PowerLawRedshift

def sample_z_powerlaw(n, lamb, z_max=2.0, grid=20000, rng=None):
    rng = np.random.default_rng() if rng is None else rng

    m = PowerLawRedshift(z_max=z_max)

    z = np.linspace(0.0, z_max, grid)
    z[0] = 1e-8  # avoid exactly 0 if you like

    dataset = {"redshift": z}

    # get (normalized) probability density on the grid
    pdf = m.probability(dataset=dataset, lamb=lamb)

    pdf = np.clip(np.nan_to_num(pdf, nan=0.0, posinf=0.0, neginf=0.0), 0.0, np.inf)

    # build a CDF with proper integration over z
    dz = np.gradient(z)
    cdf = np.cumsum(pdf * dz)
    cdf /= cdf[-1]

    u = rng.random(n)
    return np.interp(u, cdf, z)

In [17]:
from bilby.core.result import read_in_result
import numpy as np

res = read_in_result("analyses/PowerLawPeak/o1o2o3_mass_c_iid_mag_iid_tilt_powerlaw_redshift_result.json")

# pick the right one after inspecting cands
lam_samples = res.posterior['lamb'].to_numpy()

rng = np.random.default_rng(1)
lam_draw = rng.choice(lam_samples)          # one draw (or draw one per event)
z = sample_z_powerlaw(1000, lamb=lam_draw, z_max=2.0, rng=rng)

In [18]:
z

array([1.95821561, 0.95063755, 1.95666548, 1.26199084, 1.4178229 ,
       1.84983664, 1.39942129, 1.56958743, 0.53551835, 1.7804548 ,
       1.55663702, 1.28885181, 1.81351195, 1.24872674, 1.45605429,
       0.92617846, 1.39138881, 1.07703856, 1.18289965, 1.77743669,
       1.21269388, 1.49478609, 1.98387331, 1.96774795, 1.75268153,
       1.56013851, 1.20698817, 0.98843225, 1.97475463, 1.53125855,
       0.87932936, 1.65000059, 1.80247459, 1.63890012, 1.92964652,
       0.60560692, 1.54572111, 1.46329403, 0.70779942, 1.66866618,
       1.87245352, 1.61738802, 1.17917308, 1.86092683, 1.52359295,
       1.52522189, 1.7799923 , 0.95947423, 1.84243818, 1.71155352,
       1.81226462, 1.05370482, 1.82650157, 1.05311831, 0.77704781,
       1.8747881 , 1.88022509, 1.89383566, 1.47872311, 1.20234733,
       0.34019688, 1.67322177, 1.74790779, 1.85700917, 1.215065  ,
       1.09946419, 1.6665901 , 1.82899668, 1.96945707, 0.96551035,
       1.49120403, 1.90990542, 1.41703578, 1.61366315, 0.51459

In [25]:
import h5py

path = "analyses/PowerLawPeak/o1o2o3_mass_c_iid_mag_iid_tilt_powerlaw_redshift_mass_data.h5"
with h5py.File(path, "r") as f:
    print("keys:", list(f['lines'].keys()))
    print("ppd shape:", f["ppd"].shape)
    print("file attrs:", dict(f.attrs))
    print("ppd attrs:", dict(f["ppd"].attrs))

keys: ['mass_1', 'mass_ratio']
ppd shape: (500, 1000)
file attrs: {'CLASS': np.bytes_(b'GROUP'), 'DEEPDISH_IO_VERSION': np.int64(12), 'PYTABLES_FORMAT_VERSION': np.bytes_(b'2.1'), 'TITLE': Empty(dtype=dtype('S1')), 'VERSION': np.bytes_(b'1.0')}
ppd attrs: {'CLASS': np.bytes_(b'CARRAY'), 'TITLE': Empty(dtype=dtype('S1')), 'VERSION': np.bytes_(b'1.1')}


In [23]:
pop_result.posterior["lamb"]

0        0.593878
1        3.899003
2       -0.096742
3        3.511903
4        2.514446
           ...   
11179    2.733047
11180    3.171738
11181    3.171738
11182    3.456785
11183    3.456785
Name: lamb, Length: 11184, dtype: float64

In [19]:
import h5py

path = "analyses/PowerLawPeak/o1o2o3_mass_c_iid_mag_iid_tilt_powerlaw_redshift_mass_data.h5"
with h5py.File(path, "r") as f:
    print("keys:", list(f.keys()))
    print("ppd shape:", f["ppd"].shape)
    print("file attrs:", dict(f.attrs))
    print("ppd attrs:", dict(f["ppd"].attrs))

keys: ['injected', 'lines', 'ppd']
ppd shape: (500, 1000)
file attrs: {'CLASS': np.bytes_(b'GROUP'), 'DEEPDISH_IO_VERSION': np.int64(12), 'PYTABLES_FORMAT_VERSION': np.bytes_(b'2.1'), 'TITLE': Empty(dtype=dtype('S1')), 'VERSION': np.bytes_(b'1.0')}
ppd attrs: {'CLASS': np.bytes_(b'CARRAY'), 'TITLE': Empty(dtype=dtype('S1')), 'VERSION': np.bytes_(b'1.1')}
